In [2]:
# import galstreams
# mws = galstreams.MWStreams(verbose=False)

import sys, pickle, os
from importlib import reload
from tqdm import tqdm, trange

import numpy as np, pandas as pd

import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm, LogNorm

import agama 
Gyr_to_AgamaTime = 1.0227 # 1 Gyr in Agama time units (kpc/(km/s))
import nbody_streams.agama_helper as ah

import emcee
import corner

from astropy import units as u
import astropy.coordinates as coord
from astropy.coordinates import Galactocentric, ICRS, CartesianDifferential, CartesianRepresentation
from astropy import table

In [3]:
os.getcwd()

'/astro/users/shriyp/mw_gpotential_streams/notebooks'

In [4]:
sys.path.append('../scripts/')
from coordinate_utils import get_rotation_matrix, sf_to_icrs, icrs_to_sf
from generate_sim_stream import create_stream_particle_spray
from stream_likelihood import log_likelihood, make_spline, log_prior, log_probability
from stream_data_utils import read_in_data

In [5]:
# Define file paths
BASE_POT_PATH = '../potential_files'
potMW_path = os.path.join(BASE_POT_PATH, 'McMillan17_nora.ini')
potLMC_path = os.path.join(BASE_POT_PATH, 'LMC_nora.ini')
accMW_path = os.path.join(BASE_POT_PATH, 'accMW')
trajLMC_path = os.path.join(BASE_POT_PATH, 'trajLMC')

## potential models to load
potMW = agama.Potential(file=potMW_path)
accMW = np.loadtxt(accMW_path)
trajLMC = np.loadtxt(trajLMC_path)
potacc  = agama.Potential(type='UniformAcceleration', file=accMW)
potLMC  = agama.Potential(file=potLMC_path)
potLMCm = agama.Potential(potential=potLMC, center=trajLMC)
potTotal= agama.Potential(potMW, potLMCm, potacc)

In [6]:
aau_member_path = '../data/aau_members.csv'
aau_distance_path = '../data/aau_bhb_rrl.csv'

aau_table = pd.read_csv(aau_member_path)
aau_bhb_rrl_data = pd.read_csv(aau_distance_path)

aau_bhb_rrl_data.columns = aau_bhb_rrl_data.columns.str.lower()

df_aau, prog_pars_aau, prog_pars_icrs_aau, df_distance_aau = read_in_data(aau_table, 'distance_modulus', aau_bhb_rrl_data)

In [7]:
print(prog_pars_aau[2])
prog_pars_aau[2] = 10**((prog_pars_aau[2] - 10) / 5)
prog_pars_icrs_aau[2] = 10**((prog_pars_icrs_aau[2] - 10) / 5)

phi1_pos = 0
prog_pars_aau[0] = phi1_pos  #FIXING PHI1 @ specified location on stream-track

print(prog_pars_aau)

16.744050585569173
[0, 0.8556910645897031, 22.325958704826597, -0.1891601021018523, -0.9768522127958916, -114.15299941957676]


In [8]:
data_dict_aau = dict(
    phi1_obs = df_aau['phi1'].values,
    phi1_obs_dist = df_distance_aau['phi1'].values,
    phi2_obs = df_aau['phi2'].values,
    rv_obs = df_aau['vel_calib'].values,
    rv_obs_errors = df_aau['vel_calib_std'].values,
    dist_obs = 10**((df_distance_aau['distance_modulus'].values - 10) / 5),
    dist_obs_errors = 10**((df_distance_aau['distance_modulus'].values - 10) / 5) * 0.1,
    pmra_cosdec_obs = df_aau['pmra'].values,
    pmra_cosdec_obs_errors = df_aau['pmra_error'].values,
    pmdec_obs = df_aau['pmdec'].values,
    pmdec_obs_errors = df_aau['pmdec_error'].values,
)

In [9]:
#Constructing stream progenitor present-day coordinates and information

prog_mass, prog_scaleradius =  20_000, 10/1_000 # Msun, kpc

num_particles = 2_000

Age_stream_inGyr = 5.0

R = np.array([[ 0.83282408,  0.30044863, -0.46490286],
 [-0.52284461,  0.70275989, -0.48245419],
 [ 0.18176238,  0.64487142,  0.74236331]])

aau_rot_matrix = u.Quantity(R, unit=u.dimensionless_unscaled)
#aau_rot_matrix = get_rotation_matrix('AAU-ATLAS-L21', mws = mws)

print(aau_rot_matrix)


[[ 0.83282408  0.30044863 -0.46490286]
 [-0.52284461  0.70275989 -0.48245419]
 [ 0.18176238  0.64487142  0.74236331]]


In [10]:
print(data_dict_aau['phi1_obs_dist'].min(), data_dict_aau['phi1_obs_dist'].max())
phi1_obs_sel_dist = (data_dict_aau['phi1_obs_dist'] > -20) & (data_dict_aau['phi1_obs_dist'] < 20)
print(phi1_obs_sel_dist.sum())

-29.12906580105168 17.378968922864132
33


In [11]:
print(data_dict_aau['dist_obs'].shape)
print(data_dict_aau['phi1_obs'].shape)

(40,)
(140,)


In [12]:
# print("dist_obs:", data_dict_aau['dist_obs'])
# print("rv_obs range:", data_dict_aau['rv_obs'].min(), data_dict_aau['rv_obs'].max())
# ll = log_likelihood(prog_pars_aau[1:], **data_dict_aau, pot=potTotal,
#                     prog_mass=prog_mass,
#                     prog_scaleradius=prog_scaleradius,
#                     Age_stream_inGyr=Age_stream_inGyr,
#                     num_particles=num_particles,
#                     rotation_matrix=aau_rot_matrix)
# print("ll:", ll)

In [13]:
print(data_dict_aau.keys())

dict_keys(['phi1_obs', 'phi1_obs_dist', 'phi2_obs', 'rv_obs', 'rv_obs_errors', 'dist_obs', 'dist_obs_errors', 'pmra_cosdec_obs', 'pmra_cosdec_obs_errors', 'pmdec_obs', 'pmdec_obs_errors'])


In [14]:
#MCMC fit
nwalkers = 200
niter = 2500

#Taking a fraction of the prior range 
initial = prog_pars_aau[1:]
ndim = len(initial)
p0 = [np.array(initial) + 1e-2 * np.random.randn(ndim) for i in range(nwalkers)]

data = (data_dict_aau, potTotal, prog_mass, prog_scaleradius, Age_stream_inGyr, num_particles, aau_rot_matrix)

def run_burnin(p0, nwalkers, ndim, log_probability, data, nburnin=500):
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=data)
    print("Running burn-in...")
    p0, _, _ = sampler.run_mcmc(p0, nburnin, progress=True)
    print(f"Mean acceptance fraction: {np.mean(sampler.acceptance_fraction):.3f}")
    return sampler, p0

def run_production(p0, nwalkers, ndim, log_probability, data, niter=2500):
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=data)
    print("Running production...")
    pos, prob, state = sampler.run_mcmc(p0, niter, progress=True)
    return sampler, pos, prob, state

In [15]:
print("dist bounds:", data_dict_aau['dist_obs'].min(), data_dict_aau['dist_obs'].max())
print("rv bounds:", data_dict_aau['rv_obs'].min(), data_dict_aau['rv_obs'].max())
print("phi2 bounds:", data_dict_aau['phi2_obs'].min(), data_dict_aau['phi2_obs'].max())
print("pmra bounds:", data_dict_aau['pmra_cosdec_obs'].min(), data_dict_aau['pmra_cosdec_obs'].max())
print("pmdec bounds:", data_dict_aau['pmdec_obs'].min(), data_dict_aau['pmdec_obs'].max())

dist bounds: 18.323372162635163 126.08168376659422
rv bounds: -178.2297410684116 -5.170784304132981
phi2 bounds: -2.0070366481357724 4.0985051764801845
pmra bounds: -1.1884340127489397 0.707988191621612
pmdec bounds: -1.7222427110607694 0.2550274102535231


In [16]:
print(log_probability(initial, *data))

-66435.9647004612


In [ ]:
# first burn-in
sampler_burnin, p0_burnin = run_burnin(p0, nwalkers, ndim, log_probability, data, nburnin=500)
best = sampler_burnin.flatchain[np.argmax(sampler_burnin.flatlnprobability)]
print("Best params:", best)

# reinitialize and run second burn-in
p0_new = [best + 1e-2 * np.random.randn(ndim) for _ in range(nwalkers)]
sampler_burnin2, p0_burnin2 = run_burnin(p0_new, nwalkers, ndim, log_probability, data, nburnin=500)
print("Best params:", sampler_burnin2.flatchain[np.argmax(sampler_burnin2.flatlnprobability)])

# when happy, run production
p0_final = sampler_burnin2.flatchain[np.argmax(sampler_burnin2.flatlnprobability)]
p0_prod = [p0_final + 1e-2 * np.random.randn(ndim) for _ in range(nwalkers)]
sampler, pos, prob, state = run_production(p0_prod, nwalkers, ndim, log_probability, data)

Running burn-in...


  1%|          | 3/500 [00:51<2:22:43, 17.23s/it]

In [ ]:
#pulling out the values generated by the MCMC sampler XD
samples = sampler.flatchain
prog_pars_max  = samples[np.argmax(sampler.flatlnprobability)]
print(prog_pars_max)

phi2_best, dist_best, pmra_best, pmdec_best, rv_best = prog_pars_max

prog_pars_best = np.concatenate([[0], prog_pars_max])

ra_best, dec_best = sf_to_icrs(0,phi2_best, aau_rot_matrix)

ra_best, dec_best = ra_best.item(), dec_best.item()

aau_c_best = coord.SkyCoord(
    ra=ra_best*u.degree, dec=dec_best*u.degree, distance=dist_best*u.kpc, 
    pm_ra_cosdec=pmra_best*u.mas/u.yr,
    pm_dec=pmdec_best*u.mas/u.yr,
    radial_velocity=rv_best*u.km/u.s
)

rep_best = aau_c_best.transform_to(coord.Galactocentric) # units here are kpc, km/s

prog_gal_best = np.array(
    [rep_best.x.value, rep_best.y.value, rep_best.z.value,
     rep_best.v_x.value, rep_best.v_y.value, rep_best.v_z.value]
) # units here are kpc, km/s

print(prog_pars_best, aau_c_best, prog_gal_best)

In [ ]:
#MCMC checks!
labels = ['phi2', 'dist', 'pmra', 'pmdec', 'rv']

fig = corner.corner(
    samples,
    labels=labels,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_kwargs={"fontsize": 12},
    truths=prog_pars_max,  # Show best fit
    truth_color='#168600', fig_kwargs={'figsize': (4, 4)}
)
plt.show()